# 03. district_coords.csv 전처리

## 개요

서울시 25개 자치구의 **위도(lat)** 및 **경도(lng)** 데이터셋을 생성합니다.  
기존 `gu.csv` 파일에서 자치구명(n), 위도(lat), 경도(lng) 컬럼만 추출합니다.

### 사용 원본 파일

| 파일명 | 추출 변수 |
|--------|----------|
| `gu.csv` | n(자치구명), lat(위도), lng(경도) |

### 출력 파일
- `district_coords.csv`

## 0. 라이브러리 및 경로 설정

In [ ]:
import pandas as pd
import os

BASE_DIR = os.path.dirname(os.path.abspath('__file__'))
DATA_DIR = os.path.join(BASE_DIR, 'data')
OUT_DIR  = os.path.join(BASE_DIR, 'output')
os.makedirs(OUT_DIR, exist_ok=True)

GU_LIST = [
    '종로구','중구','용산구','성동구','광진구','동대문구','중랑구',
    '성북구','강북구','도봉구','노원구','은평구','서대문구','마포구',
    '양천구','강서구','구로구','금천구','영등포구','동작구','관악구',
    '서초구','강남구','송파구','강동구'
]

print(f'DATA_DIR : {DATA_DIR}')
print(f'OUT_DIR  : {OUT_DIR}')


## 1. gu.csv 로드 및 추출

### 구조
- `n`  : 자치구명 → `district`
- `lat`: 위도
- `lng`: 경도

gu.csv에는 클러스터링 등 여러 변수가 포함되어 있으므로  
위도·경도 3개 컬럼만 추출합니다.

In [ ]:
# gu.csv: CP949 / UTF-8-sig 자동 fallback
try:
    df_gu = pd.read_csv(os.path.join(DATA_DIR, 'gu.csv'), encoding='utf-8-sig')
except UnicodeDecodeError:
    df_gu = pd.read_csv(os.path.join(DATA_DIR, 'gu.csv'), encoding='cp949')

print('원본 컬럼:', list(df_gu.columns))
print(f'원본 shape: {df_gu.shape}')
df_gu.head()


## 2. 필요 컬럼 추출 및 정리

In [ ]:
# n, lat, lng 만 추출 → district로 이름 변경
df_coords = df_gu[['n', 'lat', 'lng']].rename(columns={'n': 'district'}).copy()

df_coords = df_coords[df_coords['district'].isin(GU_LIST)].copy()
df_coords['district'] = pd.Categorical(df_coords['district'], categories=GU_LIST, ordered=True)
df_coords = df_coords.sort_values('district').reset_index(drop=True)

print(f'최종 shape: {df_coords.shape}  (기대: {len(GU_LIST)} × 3)')
print(f'결측값 합계: {df_coords.isnull().sum().sum()}')
df_coords


## 3. 검증

서울시 위도·경도 범위를 벗어나는 값이 없는지 확인합니다.  
서울 기준 위도: 37.4 ~ 37.7 / 경도: 126.7 ~ 127.2

In [ ]:
# 서울시 위도·경도 정상 범위 확인 (위도 37.4~37.7 / 경도 126.7~127.2)
lat_ok = df_coords['lat'].between(37.4, 37.7).all()
lng_ok = df_coords['lng'].between(126.7, 127.2).all()

print(f'위도 범위 정상: {lat_ok}  ({df_coords["lat"].min()} ~ {df_coords["lat"].max()})')
print(f'경도 범위 정상: {lng_ok}  ({df_coords["lng"].min()} ~ {df_coords["lng"].max()})')

if not lat_ok or not lng_ok:
    print("⚠ 범위 이탈 자치구:")
    mask = ~df_coords['lat'].between(37.4, 37.7) | ~df_coords['lng'].between(126.7, 127.2)
    print(df_coords[mask])


## 4. 출력 저장

In [ ]:
out_path = os.path.join(OUT_DIR, 'district_coords.csv')
df_coords.to_csv(out_path, index=False, encoding='utf-8-sig')
print(f'✓ 저장 완료: {out_path}')
print(f'  파일 크기: {os.path.getsize(out_path):,} bytes')
